# CTA bus ridership — exploration

Companion to `frequent_network_analysis.ipynb`, which is being rebuilt. This notebook is the
**exploration** step: understand the data and the system as a whole *before* designing any
before/after comparison.

### Ground rules

- Every filter or transformation happens in a visible cell and prints what it did.
- Nothing is dropped silently. Checks print their counts **even when the count is zero**.
- Aggregate at **week** level wherever possible; month only where a week is too short a window.

### Decisions agreed so far

| Decision | Choice |
|---|---|
| Week definition | **Mon–Sun (ISO)**, labelled by the Monday that starts it |
| Day of week | Derived from `date` itself, **not** from `daytype` |
| Role of `daytype` | Only to identify CTA-designated holidays |
| Partial weeks | **Kept and flagged**, never dropped |
| Control group (later) | All corridors outside the 20 — a starting point, held as a parameter |

### Still open

- How express / branch routes (`X49`, `53A`, …) fold into corridors.
- Before/after window lengths for the event study.
- The definition of "usual" used in the holiday section below — flagged inline.

## 0. Load and inventory

No filtering happens in this section. It reads the file and counts what is in it.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. 2020 gets its own colour because it is not comparable to anything
# else; the two recovery eras are lighter shades of it because the system has not
# returned to the pre-2020 level.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-present', 2023, 2026, '#F5BE99')]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

RAW = pd.read_csv('data/cta_bus_daily.csv', dtype={'route': str, 'daytype': str})
print(f'rows read : {len(RAW):,}')
print(f'columns   : {list(RAW.columns)}')
RAW.head()

In [ ]:
# ---------------------------------------------------------------------------
# INTEGRITY REPORT.  This cell drops nothing.  It only counts.
# ---------------------------------------------------------------------------
d = RAW.copy()
d['date']  = pd.to_datetime(d.date,  errors='coerce')
d['rides'] = pd.to_numeric(d.rides, errors='coerce')

for label, n in {
    'rows':                len(d),
    'unparseable dates':   int(d.date.isna().sum()),
    'non-numeric rides':   int(d.rides.isna().sum()),
    'rides < 0':           int((d.rides < 0).sum()),
    'rides == 0':          int((d.rides == 0).sum()),
    'null route':          int(d.route.isna().sum()),
    'null daytype':        int(d.daytype.isna().sum()),
}.items():
    print(f'{label:<28}: {n:>10,}')

print()
print(f'{"date range":<28}: {d.date.min().date()} .. {d.date.max().date()}')
print(f'{"distinct routes":<28}: {d.route.nunique():>10,}')
print(f'{"daytype values":<28}: {sorted(d.daytype.dropna().unique())}')

dup = d.duplicated(subset=['route', 'date'], keep=False)
print(f'{"duplicate (route,date) rows":<28}: {int(dup.sum()):>10,}')
if dup.any():
    g = d.loc[dup].groupby(['route', 'date'])
    print(f'{"  affected route-days":<28}: {g.ngroups:>10,}')
    print(f'{"  identical rides in dup":<28}: {int((g.rides.nunique() == 1).sum()):>10,}')
    display(d.loc[dup].sort_values(['route', 'date']).head(20))
    print(f'  (showing up to 20 of {int(dup.sum()):,} duplicate rows)')

In [ ]:
# Calendar coverage: is every day between the first and last date present?
days    = pd.date_range(d.date.min(), d.date.max(), freq='D')
missing = days.difference(pd.Index(d.date.unique()))
print(f'{"calendar days in range":<28}: {len(days):>10,}')
print(f'{"days with no rows at all":<28}: {len(missing):>10,}')
if len(missing):
    print('  ', [str(x.date()) for x in missing[:20]],
          f'... (showing up to 20 of {len(missing)})')

per_day = d.groupby('date').route.nunique()
print(f'\nroutes reporting per day:  min={per_day.min()}   '
      f'median={per_day.median():.0f}   max={per_day.max()}')

## 0b. Route names and corridors

The daily file has no route names, but its **monthly sibling** on the same portal does, along
with day-type averages we can check our own aggregation against:

[CTA – Ridership – Bus Routes – Monthly Day-Type Averages & Totals](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy)
(`bynn-gwxy`), saved to `data/cta_bus_monthly.csv`:

```
curl "https://data.cityofchicago.org/resource/bynn-gwxy.csv?$limit=50000&$order=route,month_beginning" \
     -o data/cta_bus_monthly.csv
```

Columns: `route`, `routename`, `month_beginning`, `avg_weekday_rides`, `avg_saturday_rides`,
`avg_sunday_holiday_rides`, `monthtotal`.

In [ ]:
MON = pd.read_csv('data/cta_bus_monthly.csv', dtype={'route': str},
                  parse_dates=['month_beginning'])
print(f'rows {len(MON):,}   routes {MON.route.nunique()}   '
      f'{MON.month_beginning.min().date()} .. {MON.month_beginning.max().date()}')

# Names change over time, so take each route's most recent name and list the changes.
latest = MON.sort_values('month_beginning').groupby('route').routename.last()
changed = MON.groupby('route').routename.nunique()
changed = changed[changed > 1]
print(f'\nroutes renamed at least once: {len(changed)}')
for r in changed.index:
    print(f'  {r:<5} ' + ' -> '.join(MON.loc[MON.route == r, 'routename'].unique()))

print(f'\nin daily but not monthly (no name available): '
      f'{sorted(set(d.route) - set(MON.route))}')
print(f'in monthly but not daily: {sorted(set(MON.route) - set(d.route))}')

**Corridor**, starting definition: routes sharing the same numeric root — so `49`, `X49`, `49B`
belong to corridor `49`, and `J14` to corridor `14`. This is a first cut, not a final answer;
branch and express routes do not always follow the same street, so the families printed below
are meant to be audited one at a time before any of them are actually summed.

In [ ]:
import re

def corridor(route):
    """Numeric root of a route id: X49 -> 49, J14 -> 14, 1001 -> 1001."""
    m = re.search(r'\d+', route)
    return m.group() if m else route

d['name']     = d.route.map(latest)
d['corridor'] = d.route.map(corridor)

size = d.groupby('route').rides.mean()
fam  = (d.groupby('corridor').route.unique()
          .loc[lambda s: s.map(len) > 1]
          .sort_index(key=lambda i: i.astype(int)))

print(f'corridors: {d.corridor.nunique()}   of which multi-route: {len(fam)}')
print('Read this list critically -- some of these are one street, others are not.\n')
for cor, routes in fam.items():
    print(f'  corridor {cor}')
    for r in sorted(routes, key=lambda x: -size[x]):
        print(f'      {r:<6} {size[r]:>8,.0f}/day   {latest.get(r, "(no name)")}')

### Cross-check: does our daily aggregation reproduce the published monthly averages?

`daytype` `W` excludes holidays, so a month's mean over `W` days should equal the published
`avg_weekday_rides`. Any systematic gap would mean we are reading the day types wrongly.

In [ ]:
ours = (d[d.daytype == 'W']
          .groupby(['route', pd.Grouper(key='date', freq='MS')]).rides.mean()
          .rename('ours').reset_index()
          .rename(columns={'date': 'month_beginning'}))

chk = ours.merge(MON[['route', 'month_beginning', 'avg_weekday_rides']],
                 on=['route', 'month_beginning'], how='inner')
chk['diff_pct'] = (chk.ours - chk.avg_weekday_rides) / chk.avg_weekday_rides * 100

print(f'month-route pairs compared : {len(chk):,}')
print(f'  exact to within 0.5%     : {int((chk.diff_pct.abs() < 0.5).sum()):,}')
print(f'  median |difference|      : {chk.diff_pct.abs().median():.3f}%')
print(f'  worst |difference|       : {chk.diff_pct.abs().max():.2f}%')
print('\nlargest disagreements:')
print(chk.reindex(chk.diff_pct.abs().sort_values(ascending=False).index)
         .head(8)[['route', 'month_beginning', 'ours', 'avg_weekday_rides', 'diff_pct']]
         .to_string(index=False))

## 1. Weeks

Weeks are **Mon–Sun**, labelled by the Monday that starts them, derived from `date` alone.
Each week carries `days` = how many distinct calendar days actually appear in the data, so a
short week at either end of the record is visible rather than silently reading as a dip.

In [ ]:
d['week'] = d.date - pd.to_timedelta(d.date.dt.weekday, unit='D')
d['dow']  = d.date.dt.dayofweek                 # 0=Mon .. 6=Sun, from the date itself
DOW = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

wk = (d.groupby('week')
        .agg(rides=('rides', 'sum'), days=('date', 'nunique'), routes=('route', 'nunique'))
        .reset_index())
wk['partial'] = wk.days < 7

print(f'weeks spanned            : {len(wk):,}')
print(f'partial weeks (<7 days)  : {int(wk.partial.sum())}')
print()
print(wk[wk.partial].to_string(index=False))

## 2. Total ridership by week, whole dataset

The lower panel is the number of routes reporting that week. The system total is a sum over a
route set that changes over time, so the two have to be read together.

In [ ]:
wk['era'] = wk.week.dt.year.map(era)

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 5.8), sharey=False, sharex=True,
                              gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.12})

# Eras are contiguous in time, so extend each slice by one week to close the seams.
for name in ERA_ORDER:
    idx = np.flatnonzero((wk.era == name).to_numpy())
    sl = slice(idx[0], idx[-1] + 2)
    ax.plot(wk.week[sl], wk.rides[sl], lw=1.0, color=ERA_COLOR[name], label=name)

ax.set_ylabel('rides per week')
ax.yaxis.set_major_formatter(fmt_riders)
ax.set_title('CTA bus ridership by week — entire dataset', loc='left', fontsize=11)
ax.legend(frameon=False, loc='lower left', ncol=4)

# Only draw the partial-week marker if there are any; the count is printed above either way.
if wk.partial.any():
    ax.scatter(wk.loc[wk.partial, 'week'], wk.loc[wk.partial, 'rides'],
               s=30, color=PURPLE, zorder=3, label='partial week (<7 days of data)')

ax2.plot(wk.week, wk.routes, lw=1.0, color=INK)
ax2.set_ylabel('routes\nreporting')
ax2.set_xlabel('week (Monday)')
plt.show()

## 3. Individual routes by week

Every route in the dataset, one line each. The 20 Frequent Network routes are highlighted.

Log scale, because route sizes span orders of magnitude. Nothing is excluded.

> **Note.** Only the 20 route IDs as CTA labels them are highlighted. Their express / branch
> variants (`X49`, `53A`, …) are drawn in grey with everything else, because the corridor
> question is still open.

In [ ]:
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']

in_data = set(d.route.unique())
present = [r for r in FREQ if r in in_data]
print(f'Frequent Network routes (CTA labelling) : {len(FREQ)}')
print(f'  found in the data                     : {len(present)}')
print(f'  NOT found in the data                 : {[r for r in FREQ if r not in in_data]}')

rw = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
print(f'\nroute x week matrix: {rw.shape[0]:,} weeks x {rw.shape[1]:,} routes')
print(f'empty cells (route not reporting that week): {int(rw.isna().sum().sum()):,}'
      f'  of {rw.size:,}')

In [ ]:
others = [r for r in rw.columns if r not in present]

fig, ax = plt.subplots(figsize=(11, 5.6))
ax.plot(rw.index, rw[others],  lw=0.5, color=GRAY,   alpha=0.55)
ax.plot(rw.index, rw[present], lw=0.9, color=ORANGE, alpha=0.85)
ax.set_yscale('log')
ax.set_ylabel('rides per week (log scale)')
ax.set_xlabel('week (Monday)')
ax.set_title('Every bus route by week — Frequent Network routes highlighted',
             loc='left', fontsize=11)
ax.legend(handles=[Line2D([], [], color=ORANGE, lw=1.6, label=f'Frequent Network ({len(present)})'),
                   Line2D([], [], color=GRAY,   lw=1.6, label=f'all other routes ({len(others)})')],
          frameon=False, loc='lower left')
plt.show()

### Route inventory and ridership statistics

Unit throughout is **riders per day** — the raw records are daily totals, so a route's mean is
its mean over the days it reported. Weekdays, Saturdays and Sundays are pooled here, so a
route's mean reflects its weekday/weekend mix as well as its size.

`status` marks routes that stopped reporting more than 30 days before the end of the data —
these are routes that were cut or renumbered. Nothing is filtered; all 188 routes are listed.

In [ ]:
END = d.date.max()
g = d.groupby('route')

inv = pd.DataFrame({
    'name':     g.name.first(),
    'corridor': g.corridor.first(),
    'first':    g.date.min(),
    'last':     g.date.max(),
    'days':     g.date.nunique(),
    'mean':     g.rides.mean(),
    'median':   g.rides.median(),
    'std':      g.rides.std(),
    'min':      g.rides.min(),
    'max':      g.rides.max(),
})
inv['min_date'] = d.loc[g.rides.idxmin(), ['route', 'date']].set_index('route').date
inv['max_date'] = d.loc[g.rides.idxmax(), ['route', 'date']].set_index('route').date
inv['status']   = np.where(inv['last'] >= END - pd.Timedelta(days=30),
                           'active', 'ended ' + inv['last'].dt.strftime('%Y-%m'))

inv = inv.sort_values('mean', ascending=False)
for col in ('first', 'last', 'min_date', 'max_date'):
    inv[col] = inv[col].dt.strftime('%Y-%m-%d')

print(f'routes: {len(inv)}   active: {int((inv.status == "active").sum())}   '
      f'ended: {int((inv.status != "active").sum())}')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(inv[['name', 'corridor', 'first', 'last', 'days', 'status', 'mean', 'median',
                 'std', 'min', 'min_date', 'max', 'max_date']].round(1))

Same statistics split by era. A route absent from an era simply did not run then — those cells
are blank rather than zero.

> **Deviation to flag:** you suggested `pre-2020 / 2020-2022 / 2023-present`. I split 2020 out
> on its own, because pooling it with 2021–2022 hides how different it was. Collapse the two
> middle columns if you'd rather have your original grouping.

In [ ]:
d['era'] = d.date.dt.year.map(era)

per_era = (d.groupby(['route', 'era']).rides
             .agg(['mean', 'median', 'std', 'size'])
             .rename(columns={'size': 'days'})
             .unstack('era')
             .reindex(columns=ERA_ORDER, level=1))

per_era = per_era.reindex(inv.index)            # keep the sort by overall mean
print(f'2023-present is a partial era: {d[d.era == "2023-present"].date.max().date()} is the '
      f'last date, so 2026 contributes Jan-May only.')
with pd.option_context('display.max_rows', None, 'display.width', 200):
    display(per_era.round(1))

### The same tables, as plots

Three views: how concentrated ridership is across routes, how each route's recovery compares to
its pre-2020 level, and when routes started and stopped running.

In [ ]:
FREQ_SET = set(FREQ)
is_freq = inv.index.isin(FREQ_SET)
rank = np.arange(1, len(inv) + 1)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.scatter(rank[~is_freq], inv['mean'][~is_freq], s=14, color=GRAY,
           label=f'other routes ({int((~is_freq).sum())})')
ax.scatter(rank[is_freq], inv['mean'][is_freq], s=22, color=ORANGE,
           label=f'Frequent Network ({int(is_freq.sum())})', zorder=3)
for k, r in enumerate(inv.index[:8]):                       # label the eight largest
    ax.annotate(f'{r} {inv.loc[r, "name"]}', (rank[inv.index.get_loc(r)], inv.loc[r, 'mean']),
                xytext=(8, 9 if k % 2 else -9), textcoords='offset points',
                fontsize=7, va='center')
ax.set_xlabel('rank'); ax.set_ylabel('mean riders/day')
ax.yaxis.set_major_formatter(fmt_riders)
ax.set_title('Route size, ranked — whole dataset', loc='left', fontsize=11)
ax.legend(frameon=False, loc='upper right')
plt.show()

share = inv['mean'].sort_values(ascending=False).cumsum() / inv['mean'].sum()
print(f'top 20 routes carry {share.iloc[19]:.0%} of mean daily boardings; '
      f'top 50 carry {share.iloc[49]:.0%}')

In [ ]:
# Recovery: pre-2020 mean against calendar 2025, the last complete year.
y2025 = d[d.date.dt.year == 2025].groupby('route').rides.mean()
rec = pd.DataFrame({'pre': per_era[('mean', 'pre-2020')],
                    'now': y2025,
                    'name': inv.name}).dropna(subset=['pre', 'now'])
rec['ratio'] = rec.now / rec.pre
freq_mask = rec.index.isin(FREQ_SET)

fig, ax = plt.subplots(figsize=(6.6, 6.2))
lims = [rec[['pre', 'now']].to_numpy().min() * 0.7, rec[['pre', 'now']].to_numpy().max() * 1.4]
ax.plot(lims, lims, color=INK, lw=0.9, ls=':', label='no change')
ax.plot(lims, [v * 0.5 for v in lims], color=GRAY, lw=0.9, ls='--', label='half of pre-2020')
ax.scatter(rec.pre[~freq_mask], rec.now[~freq_mask], s=16, color=GRAY)
ax.scatter(rec.pre[freq_mask], rec.now[freq_mask], s=26, color=ORANGE, zorder=3,
           label='Frequent Network')
for r in rec.ratio.nlargest(4).index.union(rec.ratio.nsmallest(4).index):
    ax.annotate(r, (rec.loc[r, 'pre'], rec.loc[r, 'now']), xytext=(5, 3),
                textcoords='offset points', fontsize=7)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('mean riders/day, pre-2020'); ax.set_ylabel('mean riders/day, 2025')
ax.set_title('Recovery by route — 2025 against pre-2020', loc='left', fontsize=11)
ax.legend(frameon=False, loc='upper left')
plt.show()

print(f'routes above pre-2020 level : {int((rec.ratio > 1).sum())} of {len(rec)}')
print(f'median recovery ratio       : {rec.ratio.median():.2f}')
print('\nstrongest and weakest:')
print(pd.concat([rec.nlargest(5, "ratio"), rec.nsmallest(5, "ratio")])
        [['name', 'pre', 'now', 'ratio']].round(2).to_string())

In [ ]:
# When each route ran. Sorted by start then end date, so cuts show up as a diagonal.
span = (d.groupby('route').date.agg(['min', 'max'])
          .join(inv[['status', 'mean', 'name']])
          .sort_values(['min', 'max']))
ended = span.status != 'active'

fig, ax = plt.subplots(figsize=(11, 6.0))
y = np.arange(len(span))
ax.hlines(y[~ended], span['min'][~ended], span['max'][~ended], color=BLUE, lw=1.6,
          label=f'active ({int((~ended).sum())})')
ax.hlines(y[ended], span['min'][ended], span['max'][ended], color=ORANGE, lw=1.6,
          label=f'ended ({int(ended.sum())})')
ax.scatter(span['max'][ended], y[ended], s=9, color=ORANGE, zorder=3)

ax.set_yticks([]); ax.set_ylabel(f'route  (n={len(span)}, ordered by start date)')
ax.set_xlabel('date')
ax.set_title('Route lifespans — orange routes stopped reporting', loc='left', fontsize=11)
ax.legend(frameon=False, loc='lower right', bbox_to_anchor=(1, 1.0), ncol=2)
plt.show()

print(f'the {int(ended.sum())} routes that stopped reporting, largest first:')
print(span[ended].nlargest(20, 'mean')[['name', 'min', 'max', 'mean']]
        .assign(min=lambda t: t['min'].dt.strftime('%Y-%m'),
                max=lambda t: t['max'].dt.strftime('%Y-%m'))
        .rename(columns={'min': 'first', 'max': 'last', 'mean': 'riders/day'})
        .round(0).to_string())
print(f'(showing 20 of {int(ended.sum())})')

## 4. Ridership by day of week

Three views, because "by day of week" means different things at different scales:

- **(a)** absolute mean rides per day, one line per year — dominated by the level change.
- **(b)** the same lines divided by each year's own mean, which is the test of whether the
  *shape* is stable year to year.
- **(c)** the spread within a single recent year, so the shape in (b) can be read against how
  much an individual day actually moves.

In [ ]:
day = d.groupby('date', as_index=False).rides.sum()
day['dow']  = day.date.dt.dayofweek
day['year'] = day.date.dt.year
day['era']  = day.year.map(era)

by_yr = day.pivot_table(index='year', columns='dow', values='rides', aggfunc='mean')
shape = by_yr.div(by_yr.mean(axis=1), axis=0)      # each year normalised by its own mean
YR = 2025                                          # last complete calendar year

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))

for yr in by_yr.index:
    col = ERA_COLOR[era(yr)]
    axes[0].plot(range(7), by_yr.loc[yr], color=col, lw=1.2)
    axes[1].plot(range(7), shape.loc[yr], color=col, lw=1.2)

axes[0].set_title('(a) mean rides per day', loc='left', fontsize=10)
axes[0].yaxis.set_major_formatter(fmt_riders)
axes[1].set_title("(b) shape: divided by each year's mean", loc='left', fontsize=10)
axes[1].axhline(1, color=INK, lw=0.8, ls=':')
axes[1].legend(handles=[Line2D([], [], color=ERA_COLOR[n], lw=1.6, label=n) for n in ERA_ORDER],
               frameon=False, fontsize=8, loc='lower left')

sub = day[day.year == YR]
for i in range(7):
    v = sub.loc[sub.dow == i, 'rides']
    axes[2].scatter(i + np.random.uniform(-.16, .16, len(v)), v,
                    s=7, color=BLUE, alpha=0.35, linewidths=0)
    lo, mid, hi = np.percentile(v, [10, 50, 90])
    axes[2].plot([i - .3, i + .3], [mid, mid], color=ORANGE, lw=2, zorder=3)
    axes[2].plot([i, i], [lo, hi], color=ORANGE, lw=1, zorder=3)
axes[2].set_title(f'(c) every day in {YR}: median, 10-90th pct', loc='left', fontsize=10)
axes[2].yaxis.set_major_formatter(fmt_riders)

for ax in axes:
    ax.set_xticks(range(7)); ax.set_xticklabels(DOW)
plt.show()

**On the statistic.** Max-minus-min across years is a poor summary — it is decided by whichever
single year is most extreme, which here is 2020. Standard deviation and inter-quartile range
across years are reported instead, and separately by era, so a genuinely stable shape can be
told apart from one held together by averaging.

In [ ]:
def spread(frame):
    """Across-year spread of the normalised day-of-week shape."""
    return pd.DataFrame({
        'std':     frame.std(),
        'IQR':     frame.quantile(.75) - frame.quantile(.25),
        'max-min': frame.max() - frame.min(),
    }).rename(index=lambda i: DOW[i]).T

print('all years (2001-2026)')
print(spread(shape).round(3).to_string())

for name in ERA_ORDER:
    yrs = [y for y in shape.index if era(y) == name]
    print(f'\n{name}  (n={len(yrs)} years)')
    print('  single year - no across-year spread defined' if len(yrs) < 2
          else spread(shape.loc[yrs]).round(3).to_string())

## 5. Holidays

`daytype` is CTA's own label for which schedule ran: `W` weekday, `A` Saturday, `U` Sunday.
The true day of week comes from `date`. **A date whose label disagrees with its actual day of
week is a CTA-designated holiday** — a Monday run on a Sunday schedule.

Detecting them this way gives CTA's *operational* holiday list, which is the one that matters
for ridership. Names then come from `pandas`' `USFederalHolidayCalendar`, which also tells us
which federal holidays CTA does **not** treat as holidays.

In [ ]:
# Do all routes agree on the daytype for a given date?
lab = d.groupby('date').daytype.agg(n='nunique', label='first')
print(f'dates where routes disagree on daytype : {int((lab.n > 1).sum()):,}')

cal = lab.reset_index()[['date', 'label']]
cal['dow']      = cal.date.dt.dayofweek
cal['expected'] = np.where(cal.dow <= 4, 'W', np.where(cal.dow == 5, 'A', 'U'))
cal['holiday']  = cal.label != cal.expected

print(f'dates total                            : {len(cal):,}')
print(f'dates flagged as holiday               : {int(cal.holiday.sum()):,}')
print(f'  weekday run on a Sunday schedule     : '
      f'{int(((cal.dow <= 4) & (cal.label == "U")).sum()):,}')
print(f'  weekday run on a Saturday schedule   : '
      f'{int(((cal.dow <= 4) & (cal.label == "A")).sum()):,}')
print(f'  weekend date NOT on its own schedule : '
      f'{int(((cal.dow >= 5) & cal.holiday).sum()):,}')

h = cal[cal.holiday].copy()

# Weekend dates that did not run their own schedule -- odd enough to look at individually.
odd = h[h.dow >= 5]
print(f'\nweekend dates on an unexpected schedule ({len(odd)}):')
print(odd[['date', 'label', 'expected']].to_string(index=False))

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

# Look the names up rather than inferring them from calendar position. The calendar
# already applies the federal "observed" shift, which is what CTA follows too.
fed = USFederalHolidayCalendar().holidays(d.date.min(), d.date.max(), return_name=True)

h['name'] = h.date.map(fed)
print(f'matched on the federal OBSERVED date : {int(h.name.notna().sum())} of {len(h)}')

# The calendar lists the observed date, which shifts to the nearest weekday when a
# fixed-date holiday lands on a weekend. CTA sometimes runs the holiday schedule on the
# true date instead, so fall back to matching the actual month/day.
FIXED_DATE = {(1, 1):  "New Year's Day",
              (6, 19): 'Juneteenth National Independence Day',
              (7, 4):  'Independence Day',
              (11, 11): 'Veterans Day',
              (12, 25): 'Christmas Day'}

miss = h.name.isna()
if miss.any():
    h.loc[miss, 'name'] = [FIXED_DATE.get((t.month, t.day)) for t in h.loc[miss, 'date']]
    print(f'matched on the true (unshifted) date  : {int(miss.sum() - h.name.isna().sum())}')
    print(h.loc[miss, ['date', 'label', 'expected', 'name']].to_string(index=False))

still = h.name.isna()
print(f'\nstill unnamed: {int(still.sum())}')
if still.any():
    print(h.loc[still, ['date', 'label', 'expected']].to_string(index=False))
    h.loc[still, 'name'] = h.loc[still, 'date'].dt.strftime('%b %d') + ' (unnamed)'

print('\nflagged dates by name:')
print(h.name.value_counts().rename('dates').to_frame().to_string())

Which holidays count is not fixed in time — Juneteenth only became a federal holiday in 2021,
and `USFederalHolidayCalendar` reflects that. So the comparison below is **by year**, not a
single set difference, which would have hidden the change.

Only weekday occurrences are detectable: a holiday falling on a weekend already runs the
Saturday or Sunday schedule, so there is no label to disagree with.

In [ ]:
fed_wd = (fed.rename('name').rename_axis('date').reset_index()
             .assign(year=lambda t: t.date.dt.year)
             .query('date.dt.dayofweek <= 4'))                 # weekday occurrences only

flagged = set(zip(h.name, h.date.dt.year))
fed_wd['cta'] = [(n, y) in flagged for n, y in zip(fed_wd.name, fed_wd.year)]

# Both columns count DATES, not years: 2021 holds two federal New Year's Days
# (Jan 1, and Dec 31 as the observance of 2022's), so a year count would not line up.
print((fed_wd.groupby('name')
             .agg(federal_weekday_dates=('cta', 'size'),
                  first=('year', 'min'), last=('year', 'max'),
                  cta_ran_holiday_schedule=('cta', 'sum'))
             .sort_values('cta_ran_holiday_schedule', ascending=False)
             .to_string()))

In [ ]:
# ---------------------------------------------------------------------------
# PROPOSED definition of "usual", flagged for review before anything is built on it:
#   usual(date) = median system ridership on the SAME day of week within +/- 4 weeks,
#                 excluding dates that are themselves flagged holidays.
# ---------------------------------------------------------------------------
sysday = day.set_index('date').rides
hol_dates = set(cal.loc[cal.holiday, 'date'])

def usual(dt, dow, span_days=28):
    win = sysday[(sysday.index >= dt - pd.Timedelta(days=span_days)) &
                 (sysday.index <= dt + pd.Timedelta(days=span_days))]
    win = win[(win.index.dayofweek == dow) & (~win.index.isin(hol_dates))]
    return (win.median(), len(win))

rows = []
for dt, dw, nm in zip(h.date, h.dow, h.name):
    u, n = usual(dt, dw)
    rows.append({'date': dt, 'name': nm, 'actual': sysday.get(dt, np.nan),
                 'usual': u, 'n_baseline': n})
hr = pd.DataFrame(rows)
hr['ratio'] = hr.actual / hr.usual

print(f'holiday dates scored          : {len(hr):,}')
print(f'  with no usable baseline     : {int(hr.usual.isna().sum()):,}')
print(f'  baseline dates used, median : {hr.n_baseline.median():.0f} (of 8 possible)')
print(f'  baseline built from < 4 days: {int((hr.n_baseline < 4).sum()):,}')

In [ ]:
def box_with_points(ax, groups, labels, colour, ylabel, title, logy=False, names=None,
                    max_labels=3):
    """One box per group, individual observations jittered on top.

    If ``names`` is given (one array of labels per group), points outside the
    1.5*IQR fences are annotated, at most ``max_labels`` per side per group.
    Returns the outliers as a DataFrame so the full list can be printed.
    """
    ax.axhline(1, color=INK, lw=0.9, ls=':')
    bp = ax.boxplot(groups, positions=range(len(groups)), widths=0.6, showfliers=False,
                    patch_artist=True, medianprops={'color': ORANGE, 'lw': 1.8})
    for box in bp['boxes']:
        box.set(facecolor='#EFEFEF', edgecolor='#9A9A9A', lw=0.8)

    found = []
    for i, g in enumerate(groups):
        ax.scatter(i + np.random.uniform(-0.14, 0.14, len(g)), g,
                   s=10, color=colour, alpha=0.5, linewidths=0, zorder=3)
        if names is None:
            continue
        q1, q3 = np.percentile(g, [25, 75])
        fence = 1.5 * (q3 - q1)
        out = [(v, n) for v, n in zip(g, names[i]) if v < q1 - fence or v > q3 + fence]
        for v, n in out:
            found.append({'group': labels[i].split(chr(10))[0], 'name': n, 'value': v})
        low  = sorted([o for o in out if o[0] < q1], key=lambda t: t[0])[:max_labels]
        high = sorted([o for o in out if o[0] > q3], key=lambda t: -t[0])[:max_labels]
        for k, (v, n) in enumerate(low + high):
            ax.annotate(n, (i, v), xytext=(7, 6 if k % 2 else -6),
                        textcoords='offset points',
                        fontsize=7, color=INK, va='center')

    if logy:
        ax.set_yscale('log')
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc='left', fontsize=11)
    return pd.DataFrame(found)

In [ ]:
order  = hr.dropna(subset=['ratio']).groupby('name').ratio.median().sort_values().index
groups = [hr.loc[hr.name == nm, 'ratio'].dropna().to_numpy() for nm in order]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
box_with_points(ax, groups, [f'{nm}\n(n={len(g)})' for nm, g in zip(order, groups)], BLUE,
                'system rides ÷ usual for that weekday',
                'Holiday ridership vs the same weekday nearby — each point is one year')
plt.show()

print(hr.groupby('name').ratio.agg(['count', 'median', 'min', 'max']).round(3).to_string())

### Does the holiday effect depend on the route?

Same ratio, computed per route instead of system-wide, pooled across all years. A route whose
box sits well above the system median is one where the holiday matters less — or where the
holiday itself generates trips.

In [ ]:
rt = d.pivot_table(index='date', columns='route', values='rides', aggfunc='sum')

ratios, base_size = [], []
for dt, dw, nm in zip(h.date, h.dow, h.name):
    win = rt[(rt.index.dayofweek == dw) & ~rt.index.isin(hol_dates) &
             (rt.index >= dt - pd.Timedelta(days=28)) & (rt.index <= dt + pd.Timedelta(days=28))]
    base = win.median()
    ratios.append((rt.loc[dt] / base).replace([np.inf, -np.inf], np.nan).rename((nm, dt)))
    base_size.append(base.rename((nm, dt)))

pr   = pd.DataFrame(ratios)                       # (holiday, date) x route
base = pd.DataFrame(base_size)
print(f'route x holiday ratios computed : {int(pr.notna().sum().sum()):,}')
print(f'  ratio > 3 (route grew 3x+)    : {int((pr > 3).sum().sum()):,}')
print(f'  of those, usual < 100 rides/day: {int(((pr > 3) & (base < 100)).sum().sum()):,}')
print('\nExtreme ratios come from routes with a tiny baseline, so nothing is dropped —')
print('the axis below is log-scaled instead, and every route is shown.')

In [ ]:
med  = pr.groupby(level=0).median()               # holiday x route, median across years
keep = [nm for nm in order if nm in med.index]
groups = [med.loc[nm].dropna() for nm in keep]

fig, ax = plt.subplots(figsize=(10.5, 4.6))
outliers = box_with_points(
    ax, [g.to_numpy() for g in groups],
    [f'{nm}\n({len(g)} routes)' for nm, g in zip(keep, groups)], GREEN,
    "route rides ÷ that route's usual weekday",
    'Holiday effect by route — each point is one route, median across years',
    logy=True, names=[g.index.to_numpy() for g in groups])
plt.show()

print(f'routes outside the 1.5xIQR fences: {len(outliers)} '
      f'(plot labels at most 4 per side per holiday)')
with pd.option_context('display.max_rows', None):
    display(outliers.sort_values(['group', 'value']).reset_index(drop=True).round(3))

## 6. Seasonality

A before/after comparison needs to know what ridership does over a year anyway. Three things
make that harder than reading a monthly average, and this section deals with them one at a time.

**Trend contaminates a same-week ratio.** Dividing week *w* of one year by week *w* of the
previous year gives the seasonal difference *plus* that year's growth. So the trend is removed
first: each week is divided by a **centred 53-week moving average**, which is a full year wide
and symmetric around the week it describes. What is left is a multiplicative *seasonal index* —
1.0 means "typical for this year", 0.9 means "10% below the year's own level".

**Holiday weeks are not seasonal in the week-number sense.** Christmas moves between ISO weeks
51, 52 and 1 depending on the year, so a week-number average silently mixes holiday and normal
weeks. The 152 holiday dates found in §5 are used to hold those weeks out of the profile; the
holiday effect is then applied separately, where it can be measured properly.

**Which years to include is an empirical question**, not a choice to make up front. The profile
is computed per era and the eras are compared. If they agree, pool them; if they don't, only the
recent ones are admissible and the resulting loss of precision is a real constraint worth knowing
about before the comparison is designed.

In [ ]:
sw = wk[['week', 'rides', 'days', 'routes']].copy()
sw['year'] = sw.week.dt.year
sw['era']  = sw.year.map(era)
sw['woy']  = sw.week.dt.isocalendar().week.astype(int)     # ISO week number, 1..53

# A week is a holiday week if any of the 152 dates from section 5 falls inside it.
hol_mondays = (cal.loc[cal.holiday, 'date']
                  - pd.to_timedelta(cal.loc[cal.holiday, 'date'].dt.weekday, unit='D'))
sw['holiday_week'] = sw.week.isin(set(hol_mondays))

# Detrend. min_periods=53 means no partial windows, so 26 weeks at each end have no index.
sw['trend'] = sw.rides.rolling(53, center=True, min_periods=53).mean()
sw['index'] = sw.rides / sw.trend

print(f'weeks                        : {len(sw):,}')
print(f'  with a centred trend value : {int(sw["index"].notna().sum()):,} '
      f'({int(sw["index"].isna().sum())} at the two ends have no full window)')
print(f'  holiday weeks              : {int(sw.holiday_week.sum()):,}')
print(f'  usable for the profile     : {int((sw["index"].notna() & ~sw.holiday_week).sum()):,}')

woy_n = sw.woy.value_counts().sort_index()
print(f'\nISO week numbers present: {woy_n.index.min()}..{woy_n.index.max()}   '
      f'week 53 occurs {int(woy_n.get(53, 0))} times (ISO years with 53 weeks)')

In [ ]:
def profile(frame):
    """Median seasonal index by ISO week number, with quartiles and a count."""
    g = frame.groupby('woy')['index']
    return pd.DataFrame({'median': g.median(), 'q25': g.quantile(.25),
                         'q75': g.quantile(.75), 'n': g.size()})

usable = sw['index'].notna() & ~sw.holiday_week
profiles = {name: profile(sw[usable & (sw.era == name)]) for name in ERA_ORDER}

fig, ax = plt.subplots(figsize=(11, 4.6))
for name in ERA_ORDER:
    p = profiles[name]
    if p.empty:
        continue
    ax.plot(p.index, p['median'], lw=1.6, color=ERA_COLOR[name], label=f'{name}')
for name in ('pre-2020', '2023-present'):
    p = profiles[name]
    ax.fill_between(p.index, p.q25, p.q75, color=ERA_COLOR[name], alpha=0.18, linewidth=0)

ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index (1.0 = typical for that year)')
ax.set_title('Seasonal profile by era — holiday weeks held out, bands are the IQR',
             loc='left', fontsize=11)
ax.legend(frameon=False, ncol=4, loc='lower center')
plt.show()

In [ ]:
# Holding holiday weeks out leaves gaps: some ISO weeks contain a holiday in every year.
cover = pd.DataFrame({'all_weeks': profile(sw[sw['index'].notna()])['n'],
                      'non_holiday': profile(sw[usable])['n']}).fillna(0).astype(int)
missing = cover.index[cover.non_holiday == 0]

print(f'ISO week numbers with no non-holiday observation at all: {list(missing)}')
print('These cannot be estimated from the held-out profile and need the holiday')
print('adjustment from section 5 instead.\n')
print('thinnest coverage among the weeks that do survive:')
print(cover[cover.non_holiday > 0].nsmallest(8, 'non_holiday').to_string())

In [ ]:
# Do the two long eras describe the same season? Compare them week by week.
a, b = profiles['pre-2020'], profiles['2023-present']
common = a.index.intersection(b.index)
diff = (a.loc[common, 'median'] - b.loc[common, 'median'])

print(f'weeks compared          : {len(common)}')
print(f'correlation             : {np.corrcoef(a.loc[common, "median"], b.loc[common, "median"])[0, 1]:.3f}')
print(f'mean |difference|       : {diff.abs().mean():.4f}')
print(f'largest |difference|    : {diff.abs().max():.4f} at week {diff.abs().idxmax()}')
print(f'seasonal range, pre-2020    : {a["median"].min():.3f} .. {a["median"].max():.3f}')
print(f'seasonal range, 2023-present: {b["median"].min():.3f} .. {b["median"].max():.3f}')
print(f'\nweeks per era used:')
for name in ERA_ORDER:
    p = profiles[name]
    print(f'  {name:<14} {int(p["n"].sum()):>4} weeks across {p.index.size} week-numbers')

print('\nthe ten weeks where the two eras disagree most:')
print(pd.DataFrame({'pre-2020': a.loc[common, 'median'], '2023-present': b.loc[common, 'median'],
                    'diff': diff}).reindex(diff.abs().sort_values(ascending=False).index)
        .head(10).round(3).to_string())

### How much do holiday weeks matter?

The same profile computed with holiday weeks left in, against the one with them held out. The
gap is the reason they are held out rather than averaged over.

In [ ]:
with_hol = profile(sw[sw['index'].notna()])
without   = profile(sw[usable])
gap = (with_hol['median'] - without['median']).dropna()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(without.index, without['median'], lw=1.6, color=BLUE, label='holiday weeks held out')
ax.plot(with_hol.index, with_hol['median'], lw=1.3, color=ORANGE, ls='--',
        label='holiday weeks included')
ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index')
ax.set_title('Effect of leaving holiday weeks in the seasonal profile (all years)',
             loc='left', fontsize=11)
ax.legend(frameon=False)
plt.show()

print('weeks most affected by including holiday weeks:')
print(gap.reindex(gap.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

### Is one system-wide profile enough?

If routes have different seasonal shapes, a single system profile will mis-adjust individual
corridors. Each route's own profile is correlated against the system's.

In [ ]:
rwk = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
ridx = rwk / rwk.rolling(53, center=True, min_periods=53).mean()

ok_weeks = sw.set_index('week').loc[ridx.index, 'holiday_week'].to_numpy()
rlong = (ridx[~ok_weeks].stack().rename('index').reset_index()
           .assign(woy=lambda t: t.week.dt.isocalendar().week.astype(int)))

sys_prof = without['median']
corrs, ns = {}, {}
for r, grp in rlong.groupby('route'):
    p = grp.groupby('woy')['index'].median()
    common = p.index.intersection(sys_prof.index)
    ns[r] = len(grp)
    corrs[r] = np.corrcoef(p[common], sys_prof[common])[0, 1] if len(common) > 20 else np.nan

cs = pd.Series(corrs).rename('corr_with_system')
print(f'routes with a profile            : {int(cs.notna().sum())} of {len(cs)}')
print(f'  too few weeks to compare       : {int(cs.isna().sum())}')
print(f'median correlation with system   : {cs.median():.3f}')
print(f'  quartiles                      : {cs.quantile(.25):.3f} .. {cs.quantile(.75):.3f}')
print(f'  routes below 0.5               : {int((cs < 0.5).sum())}')

print('\nleast like the system:')
print(pd.DataFrame({'corr': cs, 'name': inv.name, 'riders/day': inv['mean'].round(0)})
        .dropna(subset=['corr']).nsmallest(10, 'corr').to_string())
print('\nFrequent Network routes:')
print(pd.DataFrame({'corr': cs, 'name': inv.name})
        .reindex(FREQ).dropna(subset=['corr']).sort_values('corr').round(3).to_string())


## Where this leaves us

### Established

- The data is clean: no duplicates, no gaps, no missing days, and our day-type handling
  reproduces the published monthly averages to within 0.001% (§0b).
- Day-of-week shape is stable to ~1-2% within an era (§4).
- CTA runs a holiday schedule on exactly six holidays, never on the other five federal ones,
  and the effect is large — 0.26x to 0.53x of a normal weekday, varying a lot by route (§5).
- Season is worth about **±10%** around a year's own level, and the profile is *not* the same
  across eras: pre-2020 and 2023-present correlate at only 0.70, differing most in late
  August (week 34-35) and late December (§6).

### Decisions still open

1. **Which years feed the seasonal profile.** Pooling pre-2020 with 2023-present assumes a
   stability the correlation does not support. Using 2023-present alone costs precision —
   135 usable weeks against 853.
2. **The `usual` definition in §5** (median of the same weekday within ±4 weeks) is a proposal.
3. **The corridor rule.** Shared numeric root is a starting point, but the R-prefixed routes are
   2013 Red Line reconstruction shuttles and do not belong in the corridors they would join.
4. **Routes with inverted seasonality.** 32 routes correlate below 0.5 with the system profile,
   and the downtown/express/tourist routes are actually *negative*. All 20 Frequent Network
   routes are between 0.57 and 0.97, so a system profile suits the treated group — but those
   inverted routes would sit in the control group, where they do not belong untreated.
